# Trening NER na Colabie (T4) — porównanie 3 modeli, encje PII

Fine-tuning na datasecie z encjami PII: `output/ner_dataset_new.jsonl`
(4584 próbki, **9 typów encji**: PERSON, CHOROBA, LEK, BADANIE, SZPITAL
+ nowe: ADRES, DATA, PESEL, TELEFON → 19 klas IOB2).

| Model | Rozmiar | Oś porównania |
|---|---|---|
| `allegro/herbert-base-cased` | 124M | baseline PL (BERT) |
| `sdadas/polish-roberta-base-v2` | 124M | BERT vs RoBERTa |
| `xlm-roberta-base` | 278M | polski vs multilingual |

**Przed startem:** Runtime → Change runtime type → **T4 GPU**.

Wyniki lecą do W&B (projekt `nlp-ner`); nazwa runa zawiera nazwę datasetu,
żeby odróżnić runy 9-typowe od starych 5-typowych.
Orientacyjny czas: ~5–10 min na model.

In [ ]:
!nvidia-smi

In [ ]:
# idempotentne: re-run komórki nie zagnieżdża klonów (ścieżki absolutne)
import os
if not os.path.exists("/content/nlp-ner"):
    !git clone -b feat/ner-training https://github.com/marek-olejniczak/nlp-ner.git /content/nlp-ner
%cd /content/nlp-ner
!git pull

In [ ]:
# torch jest preinstalowany na Colabie
!pip install -q transformers datasets seqeval accelerate wandb tqdm

In [ ]:
import wandb
wandb.login()  # wklej API key z https://wandb.ai/authorize

In [ ]:
# (model, batch, grad_accum) — wszystkie base'y mieszczą batch 16 na 16GB T4
MODELS = [
    ("allegro/herbert-base-cased", 16, 1),
    ("sdadas/polish-roberta-base-v2", 16, 1),
    ("xlm-roberta-base", 16, 1),
]

## Trening + ewaluacja w pętli

Każdy model: fine-tuning (3 epoki, lr 2e-5, fp16, najlepszy checkpoint wg F1
na walidacji) → ewaluacja na odłożonym test splicie (10%, model go nie widział).
Paski tqdm pokazują postęp treningu i ewaluacji na bieżąco.

In [ ]:
DATA = "output/ner_dataset_new.jsonl"

for model, bs, accum in MODELS:
    short = model.split("/")[-1]
    print(f"\n{'='*70}\n  {model}\n{'='*70}")
    !python -m training.train --model {model} --data {DATA} --batch-size {bs} --grad-accum {accum} --fp16
    !python -m training.evaluate --checkpoint models/{short}/best --data {DATA}

## Tabela zbiorcza (test split)

Micro F1 = wszystkie encje do jednego worka (dominują częste klasy).
Macro F1 = średnia po typach encji (wrażliwa na słabe klasy, np. LEK).

In [ ]:
import json
from pathlib import Path

print(f"{'model':35s} {'micro F1':>9s} {'macro F1':>9s}")
print("-" * 56)
for model, _, _ in MODELS:
    short = model.split("/")[-1]
    path = Path(f"models/{short}/best/eval_report_test.json")
    if not path.exists():
        print(f"{short:35s} (brak raportu)")
        continue
    report = json.loads(path.read_text())
    micro = report["micro avg"]["f1-score"]
    macro = report["macro avg"]["f1-score"]
    print(f"{short:35s} {micro:9.4f} {macro:9.4f}")

## Pobranie wytrenowanych modeli

Sesja Colaba jest ulotna — pobierz checkpointy (i raporty JSON) zanim znikną.
(~500MB na model base; alternatywnie Google Drive albo `huggingface_hub`.)

In [ ]:
# pobierz zwycięski model — podmień nazwę wg tabeli wyżej
BEST = "herbert-base-cased"
!zip -rq {BEST}-ner.zip models/{BEST}/best
from google.colab import files
files.download(f"{BEST}-ner.zip")